# Module 3: Engine
> ~25 min

In Module 2 you sent the agent a batch of conversations and turned on **Engine**. Engine has been reading those traces since then, doing the work you would otherwise do by hand: spotting failures that repeat, tracing each to its cause, and turning the failures into test cases. This module opens what it found: the issue, the evidence behind it, the fix it proposes, and the dataset examples it can write for you.

## Setup
This cell loads your keys and looks up the tracing project behind your agent's `development` environment, which is where Engine files its issues. On a project-based workspace (`LANGSMITH_PROJECT`), it uses that project instead.

> **Requires:** Engine enabled for your org and turned on for `development` in Module 2. **Without it:** read sections 2–4 against the screenshot in section 2, then do section 7.

In [ ]:
import sys
from pathlib import Path

# notebooks run from modules/; utils/ lives at the repo root
project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from langsmith import Client

from utils.config import AGENT_NAME, ENGINE_EXAMPLES_DATASET, environment_project
from utils.notebook import engine_issue_url, engine_url, trace_link
from utils.workspace import tag_to_agent

client = Client()
dest = environment_project(client)

print("Engine reads:", dest.project_name)
trace_link(engine_url(dest.url), "Engine for your agent's development environment")

## 1. Engine's picture of your agent
Before Engine files any issue, it writes an **agent overview**: what your agent does, its tools, and what to watch for. Every diagnosis is checked against that overview and the priorities you picked in Module 2. Engine drafts the overview for you, so you correct a page instead of writing a spec.

Open **Engine** in the left nav (or the link above) and click **Configure Engine** (gear). Read **Agent overview** and correct anything wrong; **Priorities** shows the categories and the custom priority from Module 2. If you skipped the repository in Module 2, connect your fork of this repo under **Code repository** now.

> **Requires:** a GitHub fork of this repo with the LangSmith GitHub App installed. **Without it:** skip the repository; Engine still diagnoses from traces, and section 4 uses **Copy Fix Context** instead of **Open PR**.

<img src="../images/03-configure-engine.png" alt="Configure Engine panel" style="width: auto; max-height: 380px; border-radius: 8px;">

The overview should name `get_client_profile` and the research subagent; if it doesn't, add them. That is how Engine knows a client lookup that never returns a profile is a failure, not expected behaviour.

## 2. The issue Engine found
Engine groups traces that fail the same way into one **issue**, with a title, a priority, a category tag and the traces behind it. You triage one issue instead of reading thirty failing traces. The cell below lists the open issues from the SDK, the same list as the **Engine** tab, with each one's priority and trace count.

In [ ]:
# the API returns severity 0-3; the UI shows these labels
PRIORITY = {0: "urgent", 1: "high", 2: "medium", 3: "low"}

issues = client.list_project_issues(dest.project_name, status="open")

print("Open issues:", len(issues))
for item in issues:
    traces = item["traces"] or []
    print(f"[{PRIORITY.get(item['severity'])}] {item['name']} ({len(traces)} traces)")

Engine writes its own titles, so the next cell finds the client-lookup issue by matching words in its title and description.

In [ ]:
LOOKUP_WORDS = ("get_client_profile", "client lookup", "client profile", "client id")

issue = None
for item in issues:
    text = f"{item['name']} {item['description']}".lower()
    if any(word in text for word in LOOKUP_WORDS):
        issue = item
        break

print("Client lookup issue:", issue["name"] if issue else "not found yet")
if issue:
    trace_link(engine_issue_url(dest.url, issue["id"]), "Open the issue")

One issue describes client lookups that fail again and again across many conversations, with a trace count close to the client-lookup threads you sent in Module 2. If the cell printed `not found yet` but you can see the client-lookup issue in the list above, Engine titled it with words the cell didn't look for. Pick it by its place in the list instead: in a new cell, run `issue = issues[0]` if it's the first row, `issues[1]` if it's the second, and so on.

> If it prints `Open issues: 0`, the first scan isn't done yet (about 20 minutes after Module 2's turn-on). Start section 7, then rerun the two cells above. The screenshot below shows what the issue looks like.

<img src="../images/03-client-lookup-issue.png" alt="The client lookup issue on the Engine tab" style="width: auto; max-height: 380px; border-radius: 8px;">

The same list is one CLI command away. The cell below prints it as a table.

In [ ]:
!langsmith projects issues list --project-id {dest.project_id} --status open --format pretty

Your coding agent can run the same command, or `langsmith projects issues get <issue-id>`, to pull one issue into its own context.

## 3. The evidence
Each issue lists **Evidence**: the traces that support the diagnosis, each with a note on why it counts. **Open trace** opens the exact child run that failed, not just the top of the trace, so you check the diagnosis without hunting for the failing step.

Click the issue on the **Engine** tab, then **Open trace** on two rows under **Evidence**. Then press `Cmd+I` and ask Chat "Why was this flagged?".

> **Requires:** a model provider for this workspace (Module 0) for Chat. **Without it:** skip the question; the evidence rows tell the same story.

<img src="../images/03-chat-why-flagged.png" alt="Chat open next to an Engine issue" style="width: auto; max-height: 380px; border-radius: 8px;">

The cell below prints the first three evidence traces with Engine's note on each.

In [ ]:
evidence = issue["traces"] if issue else []

print("Evidence traces:", len(evidence))
for link in evidence[:3]:
    print(link["trace_id"], "-", link["comment"] or "(no note)")

Each row is a client-lookup turn from Module 2: the agent passes a `client_id`, the tool rejects it, and the agent guesses again or gives up. It is the loop you found by hand in Module 2, now counted across every thread.

## 4. The proposed fix
**Proposed Fix** explains the root cause and the change that removes it. With a connected repository, it points at the lines in your code, so you start from a diagnosis rather than a stack of traces. The cell below prints it.

In [ ]:
proposed_fix = issue["proposed_fix"] if issue else None

print("Proposed fix:")
print(proposed_fix[:1500] if proposed_fix else "(none yet: rerun after the first scan)")

Expect Engine to trace the loop to `get_client_profile`'s docstring: it asks for a name slug, but profiles are keyed by IDs like `CL-0042`, and the agent has no way to map one to the other. The fix corrects the docstring or adds a name-to-ID lookup.

On the issue, click **Open PR**, then **View PR** to read the diff on your fork. Without a fork, click **Copy Fix Context** and paste it into your coding assistant.

> **Requires:** the fork connected in section 1. **Without it:** use **Copy Fix Context**.

<img src="../images/03-proposed-fix.png" alt="Proposed Fix with Open PR" style="width: auto; max-height: 380px; border-radius: 8px;">

*Example from another agent*

The **Proposed Fix** section with **Open PR**.

Once the PR is open, the cell below re-reads the issue and prints the pull requests linked to it.

In [ ]:
# fetch again: Open PR links the fix after the first read
fixes = []
for item in client.list_project_issues(dest.project_name):
    if issue and item["id"] == issue["id"]:
        fixes = item["fixes"] or []

print("Pull requests:", len(fixes))
for fix in fixes:
    print(fix["repo_url"], "PR", fix["pr_number"])

> Leave the PR unmerged. Module 5 tests the same kind of fix without it, and Module 7 replays this issue's failing threads against a deployment.

## 5. Failures become offline examples
Engine also proposes **offline examples** from its evidence: the real input, the wrong output, and a reference output written as assertions about what a correct answer contains. In a dataset, they let you test any fix against the conversations that failed, without copying inputs or writing expected answers yourself. The cell below creates an empty dataset for them and files it under your agent, so it shows in the agent's **Datasets** view.

In [ ]:
# rerunnable: reuse the dataset if it already exists
if client.has_dataset(dataset_name=ENGINE_EXAMPLES_DATASET):
    dataset = client.read_dataset(dataset_name=ENGINE_EXAMPLES_DATASET)
else:
    dataset = client.create_dataset(
        ENGINE_EXAMPLES_DATASET,
        description="Failed client lookups, added from Engine's evidence.",
    )

# The SDK doesn't file datasets under an agent the way the UI does.
tag_to_agent(client, AGENT_NAME, "dataset", dataset.id)

print("Dataset:", dataset.name)
trace_link(dataset.url, "Open the dataset")

Engine attaches its proposed examples to the issue. The next cell counts them.

In [ ]:
proposed = []
if issue:
    proposed = issue["proposed_examples"] or []

print("Examples Engine proposed:", len(proposed))

Above the **Evidence** list, click **Engine proposed N examples**. In **Add as offline example**, read each row's **Input**, **Wrong Output** and **Reference Output**, then click **Add to Dataset** and pick `client-research-engine-examples`. **Edit in annotation queue** lets a reviewer adjust the assertions first.

<img src="../images/03-add-offline-examples.png" alt="Add as offline example dialog" style="width: auto; max-height: 380px; border-radius: 8px;">

The **Add as offline example** dialog.

After you add them, the cell below re-reads the dataset and prints its example count.

In [ ]:
dataset = client.read_dataset(dataset_name=ENGINE_EXAMPLES_DATASET)

print("Examples from Engine:", dataset.example_count)
trace_link(dataset.url, "Open the dataset")

Each example's input is a client-lookup request from Module 2. Module 5 runs its experiments on the golden dataset; these examples are there when you want to test a fix against the exact failures.

## 6. Engine learns from what you do
Each action on an issue is feedback. Opening a PR tells Engine the category is real, changing the priority records how severe it is, and closing an issue as **Incorrectly Flagged** with a reason lowers similar issues. On the next scan, Engine writes these as rules under **User Preferences** in the agent overview, so you tune it by triaging rather than by writing rules.

Pick another issue Engine filed (not the client-lookup one, which Module 7 uses) and change its priority, or close it as **Incorrectly Flagged** with a one-line reason. **Create in Linear** sends an issue to your tracker once Linear is connected.

<img src="../images/03-issue-triage.png" alt="Issue priority and status controls" style="width: auto; max-height: 380px; border-radius: 8px;">

After the next scan, your reason appears as a rule under **Configure Engine** → **Agent overview** → **User Preferences**. In Module 4 your `correctness` scores become another signal: each scan pulls the lowest-scoring traces for every feedback key.

## 7. Insights
**Insights** answers a different question from Engine: it groups an environment's conversations into categories, so you see what users ask for, not what is broken, without tagging conversations or building a classifier yourself. A report runs in the background and takes up to 30 minutes.

> **Requires:** Plus or Enterprise plan, an OpenAI or Anthropic key saved as a workspace secret (Insights runs on those providers, not the Gateway), and permission to create rules. **Without it:** skip this section; Engine covers the failures.

Open **Tracing** → `client-research-agent` → **Development**, click **+New** → **New Insights Report**, answer the guided questions (try "What do relationship managers ask this agent for, and which requests go unanswered?"), and click **Run job**.

<img src="../images/03-new-insights-report.png" alt="New Insights Report setup" style="width: auto; max-height: 380px; border-radius: 8px;">

Like Engine's issues, the report is readable from code. The next cell finds the newest report on this environment and prints its categories once it has finished; rerun it if it says the report is still running.

In [ ]:
from utils.engine_reports import latest_insights_job

job = latest_insights_job(client, dest.project_id)

if job is None:
    print("No Insights report yet")
elif job["status"] != "success":
    print("Insights report:", job["name"], f"({job['status']}, rerun this cell later)")
else:
    result = client.get_insights_report(id=job["id"], project_id=dest.project_id, include_runs=False)
    print("Insights report:", result.name)
    for cluster in result.clusters:
        if cluster.level == 0:  # top-level categories only
            print(f"- {cluster.name}: {cluster.num_runs} traces")

When it finishes, client lookups, research and out-of-scope requests show up as separate categories, each with its own error rate.

<img src="../images/03-insights-report.png" alt="Insights report categories" style="width: auto; max-height: 420px; border-radius: 8px;">

*Example from another agent*

## 8. Red-team a deployment
Engine's issues come from failures that already happened. **Red Teaming** sends new, synthetic requests to a deployed agent and judges the replies, to find failures that haven't happened yet, such as prompt injection or leaked instructions. On the **Engine** tab, select **Red Teaming** at the top of the left panel and click **Run red team**; the report opens in the same view when the run completes.

> **Requires:** Red Teaming (private beta, Cloud) and the staging deployment from Module 7 set as Engine's **Baseline deployment**. **Without it:** skip for now; if you do Module 7, come back here once the deployment is live.

<img src="../images/03-red-team-report.png" alt="Red Teaming report" style="width: auto; max-height: 380px; border-radius: 8px;">

*Example from another agent*

A red-team report: confirmed findings by severity, then hypotheses and the probes that tested them.

Each probe is a real request, so its thread appears in **Tracing** under the deployment's **Staging** environment. Findings stay in the report; Red Teaming doesn't create Engine issues.

The next cell pulls the latest Red Teaming report for this environment and prints each finding with its status and severity.

In [ ]:
from utils.engine_reports import latest_red_team_run

red_team = latest_red_team_run(client, dest.project_id)

if red_team is None:
    print("No Red Teaming run yet for this environment")
else:
    report = red_team.get("report") or {}
    print("Red Teaming run:", red_team["status"])
    for finding in report.get("hypotheses", []):
        # CONFIRMED findings are failures the probes reproduced
        print(f"[{finding['status']}] {finding.get('severity', '-')}: {finding['title']}")

## Recap
| Step | Where it lives |
|---|---|
| Agent overview, priorities, repository | **Engine** tab → **Configure Engine** |
| Open issues | **Engine** tab · `client.list_project_issues` · `langsmith projects issues list` |
| Evidence | issue → **Evidence** → **Open trace** · `issue["traces"]` |
| Proposed fix | **Proposed Fix** → **Open PR** or **Copy Fix Context** · `issue["proposed_fix"]` |
| Offline examples | **Engine proposed N examples** → **Add to Dataset** |
| Teach Engine | priority, **Incorrectly Flagged** → **User Preferences** |
| Ask about an issue | Chat, `Cmd+I` |
| Usage categories | **Tracing** → agent → environment → **+New** → **New Insights Report** · `client.get_insights_report` |
| Red Teaming | **Engine** → **Red Teaming** → **Run red team** (after Module 7) · `latest_red_team_run` |

**Next:** Module 4 — Online evals and Automations

**Docs:** [Engine](https://docs.langchain.com/langsmith/engine) · [Find and fix issues](https://docs.langchain.com/langsmith/engine-issues) · [Set up Engine](https://docs.langchain.com/langsmith/engine-setup) · [Connect a repository](https://docs.langchain.com/langsmith/engine-repositories) · [Chat](https://docs.langchain.com/langsmith/chat) · [Insights](https://docs.langchain.com/langsmith/insights) · [Red Teaming](https://docs.langchain.com/langsmith/engine-red-teaming) · [LangSmith CLI](https://docs.langchain.com/langsmith/langsmith-cli)